# YouNiverse: Mapping YouTube's Hidden Communities

## Project Abstract

This project aims to uncover the hidden architecture of YouTube's content landscape. Instead of relying on explicit channel categories, we build a graph based on **audience overlap**. 

The core logic is this: for each user, we find the **Top-K channels** they comment on the most. We then create edges *only* between these Top-K channels. This creates a high-signal graph of true audience affinity.

This notebook documents the end-to-end pipeline by calling scripts from our `src` directory:

1.  **Data Loading:** Ingest channel and video metadata using `src.data.load`.
2.  **Graph Construction:** Stream comments to build the Top-K co-commenter graph using `src.scripts.process_data`.
3.  **Graph Analysis:** Normalize the graph (using our new commenter counts), run community detection, and calculate metrics using `src.models.analysis`.
4.  **Results:** Analyze and visualize the resulting communities.

## 1. Setup and Imports

First, we set up our environment, adding the `src` directory to the path and importing our custom-built modules.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

# Add src to path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Import our custom modules
from src.data import data_loader 
from src.scripts import process_data as data_processor
from src.models import model_analysis as model_analyzer

print("Successfully imported src modules.")

### Configuration

We set our main analysis parameters here. These will be passed to our scripts.

In [ ]:
# --- Parameters ---
MIN_SUBSCRIBERS = 200_000
MIN_EDGE_WEIGHT = 25
MAX_COMMENT_ROWS = 150_000_000 

# --- New Top-K Parameters ---
TOP_K_PER_AUTHOR = 5
MIN_CHANS_FOR_PAIRS = 2
AUTHOR_FLUSH_THRESHOLD = 500_000

# --- Normalization Parameters ---
NORM_ALPHA = 0.3
NORM_BETA = 1.0
USE_ENGAGEMENT_METRIC = True # Use commenter counts

# --- File Paths (relative to project root) ---
CHANNEL_METADATA_PATH = "data/raw/df_channels_en.tsv" 
VIDEO_METADATA_PATH = "data/raw/yt_metadata_helper.feather"

EDGES_OUT_PATH = "data/processed/channel_edges.csv"
CHECKPOINT_PATH = "data/processed/edges_checkpoint.pkl"
STATE_PATH = "data/processed/state.json"
DICT_PATH = "data/processed/dict_o.csv"

NODES_OUT_PATH = "data/processed/chan_graph_node_metrics.csv"
COMMUNITIES_OUT_PATH = "data/processed/chan_graph_community_summary.csv"
VIZ_OUT_PATH = "reports/figures/network_viz.png"

# Ensure processed/reports directories exist
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

print("Configuration and paths set.")

## 2. Data Loading and First Insights

First, we load the metadata we need to build our helper maps.
1.  **Channel Metadata:** To get subscriber counts for filtering.
2.  **Video Metadata:** To create a lookup map from `video_id` to `channel_id`.



In [ ]:
# Load data using our loader script
dfChannels = pd.read_csv(CHANNEL_METADATA_PATH,sep='\t')
videoDf = data_loader.load_video_metadata(local_path=VIDEO_METADATA_PATH, columns=None)

'''# Create helper maps using our processing script
v2c_map = data_processor.build_video_to_channel_map(videoDf)
channel_subset_map = data_processor.get_channel_subset_map(dfChannels, MIN_SUBSCRIBERS)

del videoDf # Free up memory
'''

In [ ]:
def set_seed(seed: int = 42):
    import random
    import os
    import numpy as np
    import torch

    # Python
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # For full reproducibility (PyTorch >= 1.8)
    torch.use_deterministic_algorithms(True)

    print(f"[Seed set to {seed}]")

set_seed(42)

## 2.1 Category distribution graph


First let's compute the data from the videos according their categories

In [ ]:
statPerCat = videoDf.groupby('categories').count().reset_index()
statPerCat.replace('','No category',inplace=True)
statPerCat.head()

Let's visualize the distribution of the categories across all videos in the dataset using a pie chart

In [ ]:
import plotly.graph_objects as go

cosmic_colors = [
    "#7b2cbf", "#ff0054", "#3f37c9", "#4cc9f0", "#f9c74f", 
    "#9d4edd", "#f72585", "#4361ee", "#480ca8", "#7209b7"
]

numChannelPerCat = statPerCat.sort_values("channel_id", ascending=False)

fig = go.Figure(data=[go.Pie(
    labels=numChannelPerCat["categories"],
    values=numChannelPerCat["channel_id"],
    hole=0.6, 
    marker=dict(
        colors=cosmic_colors,
        line=dict(color='#1A1A1A', width=2)
    ),
    textposition='outside',
    textinfo='label+percent',
    textfont=dict(
        family="Arial, sans-serif", 
        size=14, 
        color="#717171"
    ),
    pull=[0.02] * len(numChannelPerCat)
)])

fig.update_layout(
    paper_bgcolor="#1A1A1A", 
    plot_bgcolor="#1A1A1A",
    showlegend=False, 
    
    title=dict(
        text="Elemental Composition of the YouNiverse",
        x=0.5,
        y=0.95,
        font=dict(family="Arial, sans-serif", size=24, color="#C4C0C0")
    ),
    
    annotations=[
        dict(
            text=f"Total Stars<br><span style='color:#FFFFFF; font-size:22px;'><b>{numChannelPerCat['channel_id'].sum():,}</b></span>",
            showarrow=False,
            font=dict(family="Arial, sans-serif", size=16, color="#605D5D"),
            x=0.5, y=0.5,
        )
    ],
    
    height=500,
    width=850,
    margin=dict(l=100, r=100, t=100, b=100),
    
    hoverlabel=dict(
        bgcolor="#333333",
        font_size=16,
        font_family="Arial, sans-serif",
        font_color="#656565",
        bordercolor="#FFFFFF" 
    )
)


fig.write_html("pie_chart.html", include_plotlyjs='cdn', full_html=False)

fig.show()

## 2.2 Youtube Evolution


Let's see how engagement on Youtube evolved throughout the years

In [ ]:
videoDf['upload_year'] = videoDf['upload_date'].dt.year
yearViews = videoDf.groupby('upload_year').agg({'display_id' : 'count','view_count' : 'sum'})

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

yearViews.reset_index(inplace=True)
bg_color = "#1A1A1A" 

font_stack = "'-apple-system', 'BlinkMacSystemFont', 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=yearViews["upload_year"],
        y=yearViews["display_id"],
        name="Number of Videos",
        mode="lines+markers",
        line=dict(color="#8b5cf6", width=4),
        marker=dict(size=8, symbol="circle", line=dict(color="#FFFFFF", width=1)),
        hovertemplate="<b>%{x}</b><br>Videos: %{y:,}<extra></extra>"
    ),
    secondary_y=False
)

# --- 2. View Count Trace ---
fig.add_trace(
    go.Scatter(
        x=yearViews["upload_year"],
        y=yearViews["view_count"],
        name="Total Views",
        mode="lines+markers",
        line=dict(color="#06b6d4", width=4),
        marker=dict(size=8, symbol="diamond", line=dict(color="#FFFFFF", width=1)),
        hovertemplate="<b>%{x}</b><br>Views: %{y:,}<extra></extra>"
    ),
    secondary_y=True
)

# --- 3. Refined Layout & Fonts ---
fig.update_layout(
    title=dict(
        text="<span style='text-shadow: 2px 2px 0 #000;'>THE EVOLUTION OF THE YOUNIVERSE</span>",
        x=0.5,
        y=0.92,
        font=dict(size=24, family=font_stack, color="#FFFFFF") # UPDATED FONT
    ),
    paper_bgcolor=bg_color,
    plot_bgcolor=bg_color,
    hovermode="x unified",

    # Global Font Update
    font=dict(
        family=font_stack, # UPDATED FONT
        size=14,
        color="#FFFFFF"
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(0,0,0,0)",
        font=dict(family=font_stack, size=12, color="#AAAAAA") # UPDATED FONT
    ),

    margin=dict(l=80, r=80, t=120, b=80),
    
    hoverlabel=dict(
        bgcolor="#111111",
        font_size=15,
        font_family=font_stack, # UPDATED FONT
        font_color="#FFFFFF",
        bordercolor="#06b6d4"
    )
)

# --- 4. Axes Styling ---
fig.update_xaxes(
    title="TIMELINE",
    title_font=dict(family=font_stack, size=14, color="#FFFFFF"), # UPDATED FONT
    showgrid=False,
    zeroline=False,
    tickfont=dict(family=font_stack, color="#888888"), # UPDATED FONT
    linecolor="#444444"
)

fig.update_yaxes(
    title_text="<span style='color:#8b5cf6'>VIDEO VOLUME</span>",
    title_font=dict(family=font_stack), # UPDATED FONT
    showgrid=True, 
    gridcolor="rgba(255,255,255,0.05)", 
    zeroline=False,
    linecolor="#8b5cf6",
    tickfont=dict(family=font_stack, color="#8b5cf6"), # UPDATED FONT
    secondary_y=False
)

fig.update_yaxes(
    title_text="<span style='color:#06b6d4'>TOTAL VIEWS</span>",
    title_font=dict(family=font_stack), # UPDATED FONT
    showgrid=False,
    zeroline=False,
    linecolor="#06b6d4",
    tickfont=dict(family=font_stack, color="#06b6d4"), # UPDATED FONT
    secondary_y=True
)

fig.write_html("timeline_evolution.html", include_plotlyjs='cdn', full_html=False)
fig.show()

### 2.3 Distribution of views per channel

Let's see if views are distributed equally among all channels

In [ ]:
viewPerChan = videoDf.groupby('channel_id')['view_count'].sum()
viewPerChan.sort_values(ascending=False,inplace=True)

How much views does the 1% of channel hold ?

In [ ]:
(viewPerChan[:round(0.01*len(viewPerChan))].sum())/(viewPerChan.sum())

Let's graph the distribution through a histogram

In [ ]:
import plotly.graph_objects as go

bg_color = "#1A1A1A" 
font_stack = "'-apple-system', 'BlinkMacSystemFont', 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"
accent_purple = "#8b5cf6"
accent_cyan = "#06b6d4"

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=viewPerChan.reset_index()['view_count'],
    name="Channel Distribution",
    marker=dict(
        color=accent_purple,
        line=dict(color="#FFFFFF", width=0.5)
    ),
    nbinsx=100,
    hovertemplate="<b>Views:</b> %{x}<br><b>Channels:</b> %{y:,}<extra></extra>"
))

fig.update_layout(
    title=dict(
        text="<span style='text-shadow: 2px 2px 0 #000;'>DISTRIBUTION OF GALACTIC RADIANCE</span>",
        x=0.5,
        y=0.92,
        font=dict(size=24, family=font_stack, color="#989898")
    ),
    paper_bgcolor=bg_color,
    plot_bgcolor=bg_color,
    hovermode="x unified",

    font=dict(
        family=font_stack,
        size=14,
        color="#ACABAB"
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(0,0,0,0)",
        font=dict(family=font_stack, size=12, color="#AAAAAA")
    ),

    margin=dict(l=80, r=80, t=120, b=80),
    
    hoverlabel=dict(
        bgcolor="#111111",
        font_size=15,
        font_family=font_stack,
        font_color="#FFFFFF",
        bordercolor=accent_cyan 
    ),
    bargap=0.05
)

fig.update_xaxes(
    title="TOTAL VIEWS",
    title_font=dict(family=font_stack, size=14, color="#7F7F7F"),
    showgrid=False,
    zeroline=False,
    tickfont=dict(family=font_stack, color="#888888"),
    linecolor="#444444"
)

fig.update_yaxes(
    title_text="<span style='color:#8b5cf6'>NUMBER OF CHANNELS</span>",
    type="log", # Essential for Power Law visualization
    title_font=dict(family=font_stack),
    showgrid=True, 
    gridcolor="rgba(255,255,255,0.05)", 
    zeroline=False,
    linecolor=accent_purple,
    tickfont=dict(family=font_stack, color=accent_purple)
)

fig.write_html("powerlaw.html", include_plotlyjs='cdn', full_html=False)
fig.show()

## 3. Graph Construction: Streaming Comments

This is the most computationally intensive step. We call our `generate_edges` script.

As this probably would take too long to run on a personal computer without a graphic card, we created a notebook to be ran on a cloud computing platform like Google colab in src/data/data_loader.ipynb that creates edges and channel dictionary. They should be put in EDGES_OUT_PATH and DICT_PATH.

This function will stream comments, process them using the **Top-K logic**, and save both `channel_edges.csv` and the new `channel_commenter_counts.csv`.

It is also possible to create the graph only for videos published during a specific range of time. That is also shown in src/data/dataloader.ipynb. This will allow us to show the graph progress through time.

In [ ]:
v2c_map = None
'''
edges_df = data_processor.generate_edges(
    v2c_map,
    EDGES_OUT_PATH,
    CHECKPOINT_PATH,
    STATE_PATH,
    DICT_PATH,
    max_rows=MAX_COMMENT_ROWS,
    top_k_per_author=TOP_K_PER_AUTHOR,
    min_chans_for_pairs=MIN_CHANS_FOR_PAIRS,
    author_flush_threshold=AUTHOR_FLUSH_THRESHOLD
)'''
edges_df = pd.read_csv(EDGES_OUT_PATH)
print(f"\nSuccessfully loaded {len(edges_df):,} edges.")
edges_df.head()

## 4. Graph Analysis Pipeline

Now we run the main analysis pipeline. Let us find communities of similar channels based on audience. 

We will normalize our graph. The problem now is that two big channels will automatically have an edge between them with a high enough weight to shadow two smaller channels with the same edge weight but which is way more representative of channel similarity proportionally to their size. 

For example, Channel 1 and Channel 2 both have 500k total commenters that we scraped from the data. The edge weight is 10k meaning 10k of these users commented both on videos from channel 1 and channel 2. Proportionally to the channel size this is not a huge value. <br/>
On the other hand we have Channel 3 and 4 which both have 10k commenters. This time the edge weight is 5k. Meaning approximately half of their audience is shared! Edge weight is then not the best indicator channel similarity <br/> <br/>
Knowing this, we normalize edge weight to get more of a "similarity score" and find more niche communities.

In [ ]:

edges_filtered, channels_indexed = model_analyzer.filter_edges(
    edges_df, dfChannels, MIN_SUBSCRIBERS, MIN_EDGE_WEIGHT
)

commenter_counts = data_loader.load_commenter_counts(DICT_PATH)

edges_normalized = model_analyzer.normalize_edges(
    edges_filtered, 
    channels_indexed, 
    commenter_counts, 
    alpha=NORM_ALPHA, 
    beta=NORM_BETA, 
    use_engagement=USE_ENGAGEMENT_METRIC
)


G = model_analyzer.build_graph(edges_normalized, channels_indexed)

LCC, communities, node_df, comm_summary = model_analyzer.find_communities(
    G, nodes_out_path=NODES_OUT_PATH, comm_out_path=COMMUNITIES_OUT_PATH
)

## 5. Results and Interpretation

Our pipeline is complete. The results are saved in `data/processed/`. We can now load them for analysis.

### Result 1: Top 10 Largest Communities

This table shows the largest "galaxies" we found, sorted by the number of channels they contain.

In [ ]:
print("Top 10 Largest Communities:")
print(comm_summary.head(10).to_string(index=False))

### Result 2: Top 10 Most Influential Channels

This table shows the top 10 channels ranked by **PageRank**. PageRank identifies channels that are connected to *other* influential channels. These are the "super-connectors" in our YouNiverse.

In [ ]:
print("Top 10 Channels by PageRank:")
print(node_df.nlargest(10, "pagerank")[
    ["name_cc", "category_cc", "subscribers_cc", "community", "degree", "pagerank"]
].to_string(index=False))

### Result 3: Deep Dive into Community Profiles

We now call our analysis script to analyze the top communities. This helps us put a name and a theme to our algorithmically-defined clusters.

In [ ]:
# Run the analysis script
model_analyzer.analyze_communities(LCC, node_df, communities, max_show=70)

### Result 4: Network Visualization

Finally, we call our visualization script to generate and save a plot of the graph.

**Nodes** are colored by their community and sized by their strength and the legend shows us the top youtuber by community.

In [ ]:
model_analyzer.visualize_network(
    LCC, 
    communities, 
    node_df, 
    n_per_comm=5,
    min_comm_nodes=500,
    viz_out_path=VIZ_OUT_PATH, 
)

In [ ]:
import networkx as nx
import numpy as np
from pyvis.network import Network
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

def visualize_core_interactive(LCC, communities, node_df, html_out_path, num_top_communities=5, n_per_comm=20, seed=42):
    print(f"🔭 Zooming into the top {num_top_communities} galaxies...")
    
    # --- 1. Identify the 'Core' Galaxies ---
    # Sort communities by size to find the dominant structures
    sorted_communities = sorted(communities, key=len, reverse=True)
    core_communities = sorted_communities[:num_top_communities]
    
    # Map nodes to their core community ID
    node2comm = {node: cid for cid, comm in enumerate(core_communities) for node in comm}
    core_nodes_all = set(node2comm.keys())
    
    # Strength & Metadata logic
    strength = {u: sum(d.get("weight_raw", 0) for _, d in LCC[u].items()) for u in core_nodes_all}
    meta = node_df.set_index("channel_id").to_dict(orient="index")

    def name_of(uid):
        nm = (meta.get(uid, {}) or {}).get("name_cc") or uid
        return str(nm) if len(str(nm)) > 0 else str(uid)

    # --- 2. Select Representative 'Stars' from Core Galaxies ---
    selected = set()
    for nodes_c in core_communities:
        # Sort by strength to find 'Black Hole' channels
        top_nodes = sorted(nodes_c, key=lambda u: strength.get(u, 0), reverse=True)
        selected.update(top_nodes[:n_per_comm])

    # Filter subgraph: only keep bonds within the core that meet the threshold
    edges_to_keep = [(u, v) for u, v, d in LCC.edges(data=True) 
                     if u in selected and v in selected and d.get("weight_raw", 0) >= 3]
    H = LCC.edge_subgraph(edges_to_keep).copy()
    H.remove_nodes_from(list(nx.isolates(H)))

    # --- 3. Radial Expansion for the Galactic Bulge ---
    # We use a higher 'k' to push the dense core nodes apart
    pos = nx.spring_layout(H, weight="weight_raw", seed=seed, k=5.0, iterations=200)
    
    center_x = np.mean([p[0] for p in pos.values()])
    center_y = np.mean([p[1] for p in pos.values()])
    
    expanded_pos = {}
    distances = {node: np.sqrt((p[0]-center_x)**2 + (p[1]-center_y)**2) for node, p in pos.items()}
    max_dist = max(distances.values()) if distances else 1.0
    
    for node, (x, y) in pos.items():
        dist = distances[node]
        # Aggressive expansion for nodes caught in the 'central cluster'
        norm_dist = dist / max_dist
        expansion_factor = 1.0 + (3.5 * (1 - norm_dist)**2)
        
        expanded_pos[node] = (center_x + (x - center_x) * expansion_factor, 
                             center_y + (y - center_y) * expansion_factor)
    
    pos = expanded_pos
    x_range = max(p[0] for p in pos.values()) - min(p[0] for p in pos.values())
    y_range = max(p[1] for p in pos.values()) - min(p[1] for p in pos.values())

    # --- 4. Interactive Export (No Physics) ---
    net = Network(height='850px', width='100%', bgcolor='#ffffff', font_color='black', cdn_resources='remote')
    net.toggle_physics(False) 

    # Colors for core galaxies only
    comm_colors = plt.cm.Set1(np.linspace(0, 1, num_top_communities))
    SPREAD = 4500 

    for node in H.nodes():
        cid = node2comm[node]
        color = mcolors.to_hex(comm_colors[cid])
        
        # Coordinate mapping
        x = ((pos[node][0] - min(p[0] for p in pos.values())) / x_range - 0.5) * SPREAD
        y = ((pos[node][1] - min(p[1] for p in pos.values())) / y_range - 0.5) * SPREAD
        
        # Attention Inequality scaling
        size = 12 + (np.log10(strength.get(node, 1) + 1) * 15)
        
        net.add_node(
            node, x=x, y=y, label=name_of(node), color=color, size=size,
            title=f"Channel: {name_of(node)}<br>Galaxy Cluster: {cid}",
            borderWidth=1.5, font={'size': 20, 'strokeWidth': 3, 'strokeColor': '#ffffff'}
        )

    for u, v, d in H.edges(data=True):
        w = d.get("weight_raw", 1)
        # Visually represent gravitational bond strength
        width = 0.8 + (np.log10(w) * 2.0)
        net.add_edge(u, v, width=width, color='rgba(150,150,150,0.25)')

    legend_html = """
    <div style="position: absolute; top: 15px; left: 15px; width: 230px; background: rgba(255,255,255,0.9); 
                border: 2px solid #333; padding: 15px; font-family: 'Segoe UI', sans-serif; z-index: 1000; border-radius: 10px;">
        <b style="font-size: 14px; display: block; margin-bottom: 10px; border-bottom: 1px solid #333;">Dominant Galactic Hubs</b>
    """
    for cid, nodes_c in enumerate(core_communities):
        best = max(nodes_c, key=lambda u: strength.get(u, 0))
        color = mcolors.to_hex(comm_colors[cid])
        name = name_of(best)
        legend_html += f'<div style="margin-bottom: 6px;"><span style="color:{color}; font-size: 18px;">●</span> <b>Hub {cid}:</b> {name}</div>'
    legend_html += "</div>"

    net.save_graph(html_out_path)
    with open(html_out_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('<body>', f'<body>{legend_html}')
    with open(html_out_path, 'w', encoding='utf-8') as f:
        f.write(content)

    print(f"✅ Core-only interactive map saved: {html_out_path}")

Let's visualize all channels first

In [ ]:
visualize_core_interactive(
    LCC, 
    communities, 
    node_df, 
    num_top_communities=52,
    n_per_comm=10,
    html_out_path='youniverse.html'
)

Let's zoom into the core

In [ ]:
visualize_core_interactive(
    LCC, 
    communities, 
    node_df, 
    num_top_communities=5,
    n_per_comm=10,
    html_out_path='core.html'
)

Let's look at some interesting examples 

In [ ]:
import networkx as nx
import numpy as np
from pyvis.network import Network
import matplotlib.pyplot as plt

def visualize_single_galaxy(LCC, communities, node_df, community_id, html_out_path,
                           n_nodes=50,
                           weight_threshold=3,
                           seed=42):
    """
    Visualizes a single galaxy (community) from the YouNiverse using Pyvis.
    
    Parameters:
    -----------
    LCC : networkx.Graph
        The largest connected component of your network
    communities : list of sets
        List where each element is a set of nodes in that community
    node_df : pd.DataFrame
        DataFrame with channel metadata (must have 'channel_id' and 'name_cc' columns)
    community_id : int
        The ID of the community to visualize (0-indexed)
    html_out_path : str
        Path to save the HTML visualization
    n_nodes : int
        Maximum number of nodes to display (top by degree)
    weight_threshold : float
        Minimum edge weight to include
    seed : int
        Random seed for layout reproducibility
    """
    
    print(f"\n🔭 Zooming into Galaxy #{community_id}...")
    print("=" * 60)
    
    if community_id >= len(communities):
        raise ValueError(f"Community {community_id} doesn't exist. Max ID is {len(communities)-1}")
    
    galaxy_nodes = communities[community_id]
    print(f"📊 Total Channels in Galaxy: {len(galaxy_nodes)}")
    
    subgraph_full = LCC.subgraph(galaxy_nodes).copy()
    
    strength = {u: sum(d.get("weight_raw", 0) for _, d in subgraph_full[u].items()) 
                for u in galaxy_nodes}
    
    total_strength = sum(strength.values())
    avg_strength = np.mean(list(strength.values()))
    

    total_possible_edges = (len(galaxy_nodes) * (len(galaxy_nodes) - 1)) / 2
    internal_density = subgraph_full.number_of_edges() / total_possible_edges if total_possible_edges > 0 else 0
    
    clustering = nx.average_clustering(subgraph_full, weight='weight_raw')
    
    meta = node_df.set_index("channel_id").to_dict(orient="index")
    
    def name_of(uid):
        nm = (meta.get(uid, {}) or {}).get("name_cc") or uid
        return str(nm) if len(str(nm)) > 0 else str(uid)
    
    def get_category(uid):
        cat = (meta.get(uid, {}) or {}).get("join_date") or "Unknown"
        return str(cat)
    
    black_hole = max(galaxy_nodes, key=lambda u: strength.get(u, 0))
    black_hole_name = name_of(black_hole)
    black_hole_strength = strength[black_hole]
    
    top_5_channels = sorted(galaxy_nodes, key=lambda u: strength.get(u, 0), reverse=True)[:5]
    
    print(f"\n⭐ BLACK HOLE: {black_hole_name}")
    print(f"   Gravitational Strength: {black_hole_strength:,.0f}")
    print(f"\n🌟 Top 5 Channels:")
    for i, channel in enumerate(top_5_channels, 1):
        print(f"   {i}. {name_of(channel)} (strength: {strength[channel]:,.0f})")
    
    print(f"\n📈 Galaxy Statistics:")
    print(f"   Total Internal Strength: {total_strength:,.0f}")
    print(f"   Average Node Strength: {avg_strength:,.2f}")
    print(f"   Internal Density: {internal_density:.4f}")
    print(f"   Clustering Coefficient: {clustering:.4f}")
    print(f"   Total Edges: {subgraph_full.number_of_edges():,}")
    
    # 8. Select top N most important nodes for visualization
    top_nodes = sorted(galaxy_nodes, key=lambda u: strength.get(u, 0), reverse=True)[:n_nodes]
    
    # 9. Filter edges within this galaxy
    edges_to_keep = [
        (u, v) for u, v, d in subgraph_full.edges(data=True)
        if u in top_nodes and v in top_nodes and d.get("weight_raw", 0) >= weight_threshold
    ]
    
    H = LCC.edge_subgraph(edges_to_keep).copy()
    H.remove_nodes_from(list(nx.isolates(H)))
    
    print(f"\n🎨 Visualization:")
    print(f"   Displaying {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")
    print(f"   (Filtered: min {weight_threshold} edge weight)")
    
    # 10. Layout: Spring layout
    pos = nx.spring_layout(H, weight="weight_raw", seed=seed, k=2.0, iterations=100)
    
    # 11. Radial expansion
    center_x = np.mean([p[0] for p in pos.values()])
    center_y = np.mean([p[1] for p in pos.values()])
    
    expanded_pos = {}
    distances = {node: np.sqrt((p[0]-center_x)**2 + (p[1]-center_y)**2) for node, p in pos.items()}
    max_dist = max(distances.values()) if distances else 1.0
    
    for node, (x, y) in pos.items():
        dist = distances[node]
        norm_dist = dist / max_dist
        expansion_factor = 1.0 + (3.0 * (1 - norm_dist)**2)
        expanded_pos[node] = (
            center_x + (x - center_x) * expansion_factor,
            center_y + (y - center_y) * expansion_factor
        )
    
    pos = expanded_pos
    SPREAD = 2000
    
    # 12. Initialize Pyvis
    net = Network(height='600px', width='100%', bgcolor='#050505', 
                  font_color='white', cdn_resources='remote')
    net.toggle_physics(False)
    
    # Single galaxy color palette
    galaxy_colors = {
        'main': '#7b2cbf',
        'highlight': '#9d4edd',
        'accent': '#c77dff'
    }
    
    for node in H.nodes():
        x_pos = float(pos[node][0] * SPREAD)
        y_pos = float(pos[node][1] * SPREAD)
        node_strength = strength.get(node, 1)
        node_size = 5 + (np.log10(node_strength + 1) * 20)
        
        # Color based on importance
        if node_strength > np.percentile(list(strength.values()), 90):
            color = galaxy_colors['highlight']
        elif node_strength > np.percentile(list(strength.values()), 70):
            color = galaxy_colors['accent']
        else:
            color = galaxy_colors['main']
        
        net.add_node(
            node,
            x=x_pos,
            y=y_pos,
            label=name_of(node),
            color={'background': color, 'border': '#ffffff', 'highlight': '#ff0054'},
            size=node_size,
            shadow={'enabled': True, 'color': color, 'size': 5, 'x': 0, 'y': 0},
            title=f"Channel: {name_of(node)}\nStrength: {node_strength:.0f}",
            borderWidth=1,
            font={'size': 16, 'color': '#ffffff'}
        )
    
    # 14. Add edges
    for u, v, d in H.edges(data=True):
        w = d.get("weight_raw", 1)
        width = 0.5 + (np.log10(w) * 1.5)
        net.add_edge(u, v, width=width, color='rgba(255, 255, 255, 0.1)')
  
    net.save_graph(html_out_path)
    
    with open(html_out_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    zoom_js = """
    <script type="text/javascript">
    setTimeout(function() {
        network.moveTo({
            scale: 0.05,
            position: {x: 0, y: 2700},
            animation: false
        });
    }, 100);
    </script>
    """
    
    style_fix = """
    <style>
    html, body {
        margin: 0 !important;
        padding: 0 !important;
        width: 100%;
        height: 100%;
        border: 0;
        background: #050505 !important;
        overflow: hidden;
    }
    .card {
        border: none !important;
        outline: none !important;
        border-radius: 0 !important;
        background-color: black;
    }
    #mynetwork {
        width: 100% !important;
        background: #050505 !important;
        border: #000000;
    }
    canvas {
        display: block !important;
        background: #050505 !important;
    }
    </style>
    """
    
    content = content.replace("</head>", style_fix + "\n</head>")
    content = content.replace("<body>", '<body style="background-color:#050505; margin:0; overflow:hidden;">')
    content = content.replace("</body>", f"{zoom_js}\n</body>")
    
    with open(html_out_path, 'w', encoding='utf-8') as f:
        f.write(content)
        f.flush()
    
    print(f"\n✅ Galaxy #{community_id} visualization saved to: {html_out_path}")
    print("=" * 60)
    
    return {
        'community_id': community_id,
        'total_channels': len(galaxy_nodes),
        'total_strength': total_strength,
        'avg_strength': avg_strength,
        'internal_density': internal_density,
        'clustering_coefficient': clustering,
        'total_edges': subgraph_full.number_of_edges(),
        'displayed_nodes': H.number_of_nodes(),
        'displayed_edges': H.number_of_edges(),
        'black_hole': black_hole_name,
        'black_hole_strength': black_hole_strength,
        'top_5_channels': [name_of(ch) for ch in top_5_channels]
    }

English learning

In [ ]:
stats = visualize_single_galaxy(
    LCC=LCC,
    communities=communities,
    node_df=node_df,
    community_id=24,
    html_out_path='galaxy_24.html',
    n_nodes=50,
    weight_threshold=3,
    seed=42
)

Tennis galaxy

In [ ]:
stats = visualize_single_galaxy(
    LCC=LCC,
    communities=communities,
    node_df=node_df,
    community_id=25,
    html_out_path='galaxy_25.html',
    n_nodes=50,
    weight_threshold=3,
    seed=42
)

Maghrebi galaxy

In [ ]:
stats = visualize_single_galaxy(
    LCC=LCC,
    communities=communities,
    node_df=node_df,
    community_id=26,
    html_out_path='galaxy_26.html',
    n_nodes=50,
    weight_threshold=3,
    seed=42
)

## 6. Topic Detection for Communities

Now we apply topic detection to identify the themes within each community. This uses LDA (Latent Dirichlet Allocation) following the methodology from Lab 9.

In [ ]:
# Install required packages if not already installed
# !pip install spacy gensim
# !python -m spacy download en_core_web_sm

import spacy

# Load spacy model
try:
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    print("✓ Spacy model loaded")
except:
    print("⚠️  Run: python -m spacy download en_core_web_sm")
    nlp = None

### Load video metadata for topic detection

In [ ]:
# Load video metadata for topic detection
# Stream directly from the YouNiverse dataset zip file
# Only loads videos from channels in our communities (efficient)

# Sort communities by size (largest first) for better analysis
communities_sorted = sorted(communities, key=len, reverse=True)
print("Top 10 communities by size:")
for i, comm in enumerate(communities_sorted[:10]):
    print(f"  Community {i}: {len(comm):,} channels")

# Collect channel IDs from the LARGEST communities
n_communities_to_analyze = 10
all_channels = set()
for i in range(min(n_communities_to_analyze, len(communities_sorted))):
    all_channels.update(communities_sorted[i])

print(f"\nNeed videos from {len(all_channels)} unique channels across {n_communities_to_analyze} largest communities")

# Stream videos from the zip file (no extraction needed)
# Load ALL fields once for both topic detection AND engagement metrics
video_df = data_loader.stream_video_metadata_from_zip(
    zip_path='data/youniverse-dataset.zip',
    channel_filter=all_channels,
    max_videos_per_channel=200,  # 200 videos per channel for topics & engagement analysis
    fields='all'  # Load all fields: title, description, tags, view_count, like_count, duration, etc.
)

if len(video_df) == 0:
    print("\nError: No videos loaded. Check that data/youniverse-dataset.zip exists.")
else:
    print(f"\n✓ Successfully loaded {len(video_df):,} videos")
    print(f"  Columns: {list(video_df.columns)}")
    print(f"  Ready for topic detection AND engagement analysis!")

### Analyze topics for each community

In [ ]:
# Analyze top communities
# Reload module to ensure latest code is used
import importlib
importlib.reload(model_analyzer)

n_communities_to_analyze = 10
community_topics = {}

# Verify video_df has the right columns
print(f"Video DataFrame columns: {list(video_df.columns)}")
print(f"Video DataFrame shape: {video_df.shape}")

# Use communities_sorted (sorted by size, largest first)
for i, community in enumerate(communities_sorted[:n_communities_to_analyze]):
    print(f"\n{'='*80}")
    print(f"COMMUNITY {i} (size: {len(community)} channels)")
    print(f"{'='*80}\n")
    
    result = model_analyzer.categoryDetect(
        community=community,
        video_metadata_df=video_df,
        k_channels=25,              # Sample 25 channels per community
        n_videos_per_channel=100,   # Up to 100 videos per channel
        n_topics=6,                 # Extract 6 topics
        text_mode='title_desc',     # Use title + truncated desc only (NO TAGS - less noise)
        desc_max_chars=300,         # More context from descriptions (URLs stripped)
        min_wordcount=5,
        max_freq=0.45,               # Let alpha/eta='auto' handle topic separation
        passes=25,                  # Good convergence
        seed=42,
        nlp=nlp
    )
    
    community_topics[i] = result
    print("\n")

### Compare detected topics with official YouTube categories

In [ ]:
# Compare detected topics with official categories
for comm_id in range(min(3, len(communities_sorted))):
    community = communities_sorted[comm_id]
    
    # Get official categories from node_df
    comm_nodes = node_df[node_df['channel_id'].isin(community)]
    categories = comm_nodes['category_cc'].value_counts()
    
    print(f"\n{'='*80}")
    print(f"Community {comm_id}")
    print(f"{'='*80}")
    
    print(f"\nOfficial YouTube Category Distribution:")
    for cat, count in categories.head(5).items():
        pct = count / len(community) * 100
        print(f"  {cat:20s} {count:4d} channels ({pct:5.1f}%)")
    
    if comm_id in community_topics and community_topics[comm_id]['topics']:
        print(f"\nDetected Topics (LDA):")
        for topic_id, topic_words in enumerate(community_topics[comm_id]['topics']):
            words = ', '.join([word for word, _ in topic_words[:5]])
            print(f"  Topic {topic_id + 1}: {words}")
    
    print()

### Visualize topics with pyLDAvis

In [ ]:
# Display topics nicely for each community
def display_topics(community_topics):
    for comm_id, result in community_topics.items():
        print(f"\n{'='*60}")
        print(f"COMMUNITY {comm_id} - Detected Topics")
        print(f"{'='*60}")
        print(f"Videos analyzed: {result.get('n_videos', 0)}")
        print(f"Channels sampled: {len(result.get('sampled_channels', []))}")
        
        topics = result.get('topics', [])
        if not topics:
            print("No topics detected")
            continue
            
        for topic_id, topic_words in enumerate(topics):
            print(f"\n  Topic {topic_id + 1}:")
            top_words = [f"{word} ({weight:.2f})" for word, weight in topic_words[:5]]
            print(f"    {', '.join(top_words)}")

display_topics(community_topics)

### Interactive Topic Visualization with pyLDAvis

In [ ]:
# Install pyLDAvis if needed (uncomment and run once)
# !pip install pyldavis

import warnings
warnings.filterwarnings('ignore')  # Suppress multiprocessing warnings

import pyLDAvis
import pyLDAvis.gensim_models

# Enable notebook visualization
pyLDAvis.enable_notebook()

# Visualize topics for a specific community
comm_id = 0  # Change this to visualize different communities (0, 1, 2, ...)

if comm_id in community_topics:
    result = community_topics[comm_id]
    if result.get('model') and result.get('corpus') and result.get('dictionary'):
        print(f"Preparing visualization for Community {comm_id}...")
        
        # Prepare the visualization data
        vis_data = pyLDAvis.gensim_models.prepare(
            result['model'], 
            result['corpus'], 
            result['dictionary'],
            sort_topics=False
        )
        
        # Save to HTML file (always works)
        html_path = f'reports/figures/lda_community_{comm_id}.html'
        pyLDAvis.save_html(vis_data, html_path)
        print(f"✓ Saved interactive visualization to: {html_path}")
        print(f"  Open this file in your browser to see the visualization!")
        
        # Try to display inline (may not work in all environments)
        try:
            pyLDAvis.display(vis_data)
        except Exception as e:
            print(f"\\n(Inline display not available: {e})")
            print("Use the HTML file instead.")
    else:
        print(f"Community {comm_id} missing model/corpus/dictionary")
else:
    print(f"Community {comm_id} not found in results")

### Save topic detection results

In [ ]:
# Create summary DataFrame
summary_data = []

for comm_id, result in community_topics.items():
    if result['topics']:
        # Get top 5 words from each topic
        topic_summaries = []
        for topic_words in result['topics']:
            top_words = ', '.join([word for word, _ in topic_words[:5]])
            topic_summaries.append(top_words)
        
        summary_data.append({
            'community_id': comm_id,
            'n_channels': len(communities_sorted[comm_id]),
            'n_videos_analyzed': result['n_videos'],
            'n_topics': len(result['topics']),
            'topics': ' | '.join(topic_summaries)
        })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('data/processed/community_topics_summary.csv', index=False)

print("✓ Results saved to data/processed/community_topics_summary.csv\n")
summary_df

## 6.5 Engagement Metrics Analysis: Behavioral Differences Across Galaxies

Now we analyze how different galaxies behave in terms of engagement metrics:
- **Video Duration**: How long are videos in each galaxy?
- **View Count**: How much attention do videos receive?
- **Like Count**: How engaged are viewers?
- **Engagement Rate**: Likes per view

This helps us understand not just *what* each galaxy is about (topics), but *how* audiences interact with content in each one.

In [ ]:
# Use the already loaded video_df (loaded with fields='all' in the topic detection section)
# No need to reload - we already have all the engagement metrics!

print("Using video_df already loaded earlier (with all fields)")
print(f"\n✓ {len(video_df):,} videos available with engagement metrics")
print(f"Columns: {list(video_df.columns)}")

# Verify engagement columns are present
engagement_cols = ['view_count', 'like_count', 'dislike_count', 'duration']
missing_cols = [c for c in engagement_cols if c not in video_df.columns]
if missing_cols:
    print(f"\n⚠️ Missing columns: {missing_cols}")
    print("Please re-run the video loading cell with fields='all'")
else:
    print("\n✓ All engagement metrics available!")

In [ ]:
# Convert engagement columns to numeric (using video_df loaded earlier)
for col in ['view_count', 'like_count', 'dislike_count', 'duration']:
    if col in video_df.columns:
        video_df[col] = pd.to_numeric(video_df[col], errors='coerce')

# Map channels to communities
channel_to_community = {}
for comm_id, community in enumerate(communities_sorted[:n_communities_to_analyze]):
    for channel in community:
        channel_to_community[channel] = comm_id

# Add community ID to video dataframe
video_df['community_id'] = video_df['channel_id'].map(channel_to_community)

# Filter to only videos from analyzed communities
videos_with_comm = video_df.dropna(subset=['community_id']).copy()
videos_with_comm['community_id'] = videos_with_comm['community_id'].astype(int)

print(f"Videos with community mapping: {len(videos_with_comm):,}")
print(f"\nSample of engagement data:")
videos_with_comm[['channel_id', 'title', 'view_count', 'like_count', 'duration', 'community_id']].head()

In [ ]:
# Compute engagement statistics per community
engagement_stats = []

for comm_id in range(n_communities_to_analyze):
    comm_videos = videos_with_comm[videos_with_comm['community_id'] == comm_id]
    
    if len(comm_videos) == 0:
        continue
    
    # Calculate metrics
    stats = {
        'community_id': comm_id,
        'n_channels': len(communities_sorted[comm_id]),
        'n_videos': len(comm_videos),
        
        # View statistics
        'mean_views': comm_videos['view_count'].mean(),
        'median_views': comm_videos['view_count'].median(),
        'total_views': comm_videos['view_count'].sum(),
        
        # Like statistics
        'mean_likes': comm_videos['like_count'].mean(),
        'median_likes': comm_videos['like_count'].median(),
        
        # Duration statistics (in seconds)
        'mean_duration_sec': comm_videos['duration'].mean(),
        'median_duration_sec': comm_videos['duration'].median(),
    }
    
    # Calculate engagement rate (likes per 1000 views)
    if stats['mean_views'] > 0:
        stats['engagement_rate'] = (stats['mean_likes'] / stats['mean_views']) * 1000
    else:
        stats['engagement_rate'] = 0
    
    # Format duration as minutes
    stats['mean_duration_min'] = stats['mean_duration_sec'] / 60 if pd.notna(stats['mean_duration_sec']) else 0
    
    engagement_stats.append(stats)

engagement_df = pd.DataFrame(engagement_stats)
print("\n=== Engagement Statistics by Community ===")
engagement_df

In [ ]:
# Visualize engagement differences across communities
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Median Views by Community
ax1 = axes[0, 0]
bars1 = ax1.bar(engagement_df['community_id'], engagement_df['median_views'], color='steelblue')
ax1.set_xlabel('Community ID')
ax1.set_ylabel('Median View Count')
ax1.set_title('Median Views by Galaxy')
ax1.set_xticks(engagement_df['community_id'])

# Plot 2: Mean Duration by Community
ax2 = axes[0, 1]
bars2 = ax2.bar(engagement_df['community_id'], engagement_df['mean_duration_min'], color='coral')
ax2.set_xlabel('Community ID')
ax2.set_ylabel('Mean Duration (minutes)')
ax2.set_title('Average Video Duration by Galaxy')
ax2.set_xticks(engagement_df['community_id'])

# Plot 3: Engagement Rate by Community
ax3 = axes[1, 0]
bars3 = ax3.bar(engagement_df['community_id'], engagement_df['engagement_rate'], color='seagreen')
ax3.set_xlabel('Community ID')
ax3.set_ylabel('Likes per 1000 Views')
ax3.set_title('Engagement Rate by Galaxy')
ax3.set_xticks(engagement_df['community_id'])

# Plot 4: Total Views (log scale)
ax4 = axes[1, 1]
bars4 = ax4.bar(engagement_df['community_id'], engagement_df['total_views'], color='purple')
ax4.set_xlabel('Community ID')
ax4.set_ylabel('Total Views (log scale)')
ax4.set_title('Total Views by Galaxy')
ax4.set_xticks(engagement_df['community_id'])
ax4.set_yscale('log')

plt.tight_layout()
plt.savefig('reports/figures/engagement_by_community.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved reports/figures/engagement_by_community.png")

In [ ]:
# Combine topic detection results with engagement metrics
# Add galaxy names based on detected topics (from our previous analysis)

# Get top category for each community from node_df
for i, row in engagement_df.iterrows():
    comm_id = int(row['community_id'])  # Convert to int for list indexing
    community = communities_sorted[comm_id]
    comm_nodes = node_df[node_df['channel_id'].isin(community)]
    if len(comm_nodes) > 0 and 'category_cc' in comm_nodes.columns:
        top_cat = comm_nodes['category_cc'].value_counts().index[0]
        engagement_df.loc[i, 'top_category'] = top_cat

# Save to CSV
engagement_df.to_csv('data/processed/community_engagement_metrics.csv', index=False)
print("✓ Saved engagement metrics to data/processed/community_engagement_metrics.csv")

# Display summary
print("\n=== Galaxy Engagement Summary ===")
display_cols = ['community_id', 'n_channels', 'top_category', 'median_views', 'mean_duration_min', 'engagement_rate']
engagement_df[display_cols].round(2)

### Key Insights from Engagement Analysis

The engagement metrics reveal behavioral differences between galaxies:

- **Video Duration**: Some galaxies (e.g., Gaming) may have longer videos, while others (e.g., Music) have shorter content
- **View Counts**: Mainstream galaxies typically have higher median views
- **Engagement Rate**: Niche communities often show higher engagement rates (more likes per view)

These patterns help distinguish galaxy "personality" beyond just thematic content.

### 7.1 Measuring how attention leaves a galaxy

Concretely, the code in this cell:
- maps each channel to its detected community,
- aggregates channel-to-channel interactions into **community-to-community flows**,
- separates internal from cross-community interactions,
- and derives several mobility indicators:
  - **external share** (how much attention leaves the community),
  - **destination diversity** (entropy / Gini),
  - **top destination share** (how concentrated exits are).

The goal is to build **quantitative signals**
that describe *how audiences move once they leave their home galaxy*.

At this stage, we expect to observe heterogeneous behaviors: some communities may keep
most attention inside, while others may act as more open systems.


In [ ]:
# Build community-level flows and mobility metrics via helper
flows = model_analyzer.compute_comm_flows(edges_normalized, node_df, comm_summary)
flow_norm = flows["flow_norm"]
flow_share = flows["flow_share"]
pair_strength = flows["pair_strength"]
communities_sorted = flows["communities_sorted"]
mobility_df = flows["mobility_df"]
mobility_df.head(10)


### 7.2 Dominant destinations — who do communities send attention to?

After measuring how much attention leaves each community, we now look at *where* this
attention goes.

In this cell, the code:
- ranks destination communities by share of outgoing flow,
- identifies each community’s top partner (and second partner),
- and highlights cases where most external attention is concentrated on a single
  destination.

This helps us detect **dependency patterns**, where communities rely heavily on a
specific external galaxy.


In [ ]:
top_partner_df = model_analyzer.top_destinations(flow_share, communities_sorted)
display(top_partner_df.head(30))


### 7.3 Global circulation patterns — identifying major inter-community corridors

We now zoom out to map the main arteries of audience circulation across the Youniverse.

This cell:
- symmetrizes community-to-community flows to focus on mutual exchange,
- keeps the 100 strongest links to avoid clutter,
- plots a chord-style map of these corridors and lists the heaviest pairs.

The goal is to surface the backbone of inter-community traffic before we dive into
directional details.


In [ ]:
model_analyzer.plot_chord_backbone(
    pair_strength,
    communities_sorted,
    top_n=300,
    out_path="reports/figures/community_chord_simple.png",
)



### 7.4 Echo-chamber candidates — combining openness and exit diversity

We blend openness (`external_share`) and destination diversity (`entropy_out`) to flag
communities that behave like closed systems.

This cell:
- sets cutoffs at the 25th percentile of external_share and the median entropy_out,
- selects communities that fall below both thresholds,
- visualizes external share with suspected echo chambers highlighted.

The output is a short watchlist of communities to inspect qualitatively.


In [ ]:
echo_candidates, q_ext, q_ent = model_analyzer.find_echo_candidates(mobility_df, 0.25, 0.50)
print(f"Seuils: external_share <= {q_ext:.3f}, entropie <= {q_ent:.3f}")
print("Communautés les plus fermées (tri multi-critères):")
display(echo_candidates.head(10)[[
    "community", "external_share", "entropy_out", "gini_out", "n_nodes", "top_category"
]])

model_analyzer.plot_echo_share(
    mobility_df,
    echo_candidates,
    q_ext,
    out_path="reports/figures/echo_chamber_external_share.png",
)


### 7.5 Bridge channels — identifying who enables cross-community travel

We now zoom in at the channel level to see who physically moves attention between
communities.

This cell:
- builds a length-weighted graph and computes sampled betweenness on the LCC,
- measures each channel's cross-community share and raw cross strength,
- flags bridge channels that send most of their weight outside their home community
  while carrying substantial cross-community volume.

These connectors are the structural portals between otherwise separate galaxies.


In [ ]:
bridge = model_analyzer.compute_bridge_channels(
    LCC,
    node_df,
    communities,
    betweenness_k=800,
    cross_share_min=0.5,
    cross_strength_min=1000,
)

print("Top canaux ponts (part vers d’autres communautés + betweenness):")
display(
    bridge["bridge_top"].head(15)[
        ["name", "community", "cross_share", "cross_strength", "betweenness", "degree"]
    ]
)


### 7.6 Bridge categories — through which genres do audiences travel?

After surfacing bridge channels, we check whether certain genres act as the main
vehicles for cross-community travel.

This cell:
- merges bridge channels with their content categories,
- ranks the top bridge channels per community,
- aggregates bridge strength by community × category to see which genres dominate.

The result reveals whether travel happens through a few recurring genres or a broad mix.


In [ ]:
from ipywidgets import interact, widgets

bridge_top_cat = bridge["bridge_top_cat"]
top_channels = bridge["top_channels"]
agg = bridge["agg"]

print("Top bridge channels per community (max 10):")
display(top_channels[[
    "community", "name", "category_cc", "cross_share", "cross_strength", "betweenness", "degree"
]])

print("Bridge categories per community:")
display(agg.head(20))


### 7.7 Inspecting bridge channels locally — zooming inside each community

We take a closer look at which channels drive cross-community traffic within each
community.

This cell:
- offers an interactive dropdown to display the top bridge channels for any community,
  sorted by cross strength and annotated with cross-share,
- renders a static multi-panel bar chart of the top bridge channels per community,
  saved for reporting.

The aim is qualitative: spot the key connectors and how their behavior varies by community.


In [ ]:
import seaborn as sns

# Interactive bar retained; static plot moved to helper
communities = sorted(top_channels["community"].unique())
palette = sns.color_palette("tab20", len(communities))
comm_color = {c: palette[i % len(palette)] for i, c in enumerate(communities)}

@interact(comm=widgets.Dropdown(options=communities, description="Community"))
def plot_top_channels_by_comm(comm):
    data = top_channels[top_channels["community"] == comm].sort_values("cross_strength", ascending=False)
    if data.empty:
        print("No bridge channels for this community.")
        return
    plt.figure(figsize=(9, max(4, 0.4 * len(data))))
    sns.barplot(
        data=data,
        y="name",
        x="cross_strength",
        color=comm_color.get(comm, "#1f77b4")
    )
    for i, row in data.reset_index().iterrows():
        plt.text(row["cross_strength"] * 1.01, i, f"share {row['cross_share']:.2f}", va="center", fontsize=7)
    plt.xlabel("Cross strength (raw weight)")
    plt.ylabel("Channel")
    plt.title(f"Top bridge channels — community {comm}")
    plt.tight_layout()
    plt.show()

model_analyzer.plot_bridge_top_channels(
    top_channels,
    topN=10,
    out_path="reports/figures/bridge_channels_static_overview.png",
)


### 7.8 Which genres connect galaxies? — bridge categories at the community level

We now summarize bridge behavior by content category to see which genres act as
connectors.

This cell:
- takes the aggregated bridge categories,
- keeps the top three categories per community by cross strength,
- plots a grouped bar chart and saves it for reporting.

This highlights the genres most responsible for moving audiences across communities.


In [ ]:
model_analyzer.plot_bridge_categories(
    agg,
    out_path="reports/figures/bridge_categories_top3.png",
)


### 7.9 Directional travel corridors — who sends attention to whom?

We bring directionality back to spotlight the strongest source → destination pairs.

This cell:
- reshapes the community flow matrix and drops self-loops,
- filters to non-zero cross-community flows,
- ranks the top 20 normalized flows, plots them, and saves the figure.

It surfaces asymmetric travel patterns and the communities acting as main senders or receivers.


In [ ]:
model_analyzer.plot_directional_flows(
    flow_norm,
    top_n=20,
    out_path="reports/figures/bridge_community_topflows.png",
)
